In [50]:
import torch
import math

device = 'cuda' if torch.cuda.is_available() else 'cpu'

batch_size, seq_len, emb_dim = 2, 5, 32

In [ ]:
class SelfAttn(torch.nn.Module):
  def __init__(self, emb_dim=32, num_heads=4):
    super().__init__()
    self.head_dim = emb_dim // num_heads
    self.emb_dim = emb_dim
    self.num_heads = num_heads

    self.attn = torch.nn.Linear(emb_dim, emb_dim * 3)
    self.output_proj = torch.nn.Linear(emb_dim, emb_dim)

  def forward(self, x):
    x = self.attn(x)
    q, k, v = torch.split(x, self.emb_dim, dim=-1)
    
    q = q.view(batch_size, seq_len, self.num_heads, self.head_dim) # reshape + permute
    k = k.view(batch_size, seq_len, self.num_heads, self.head_dim)
    v = v.view(batch_size, seq_len, self.num_heads, self.head_dim)
    
    # TODO: address padding tokens?
    
    attn_weights = (q @ k.transpose(-1, -2)) / math.sqrt(self.emb_dim)
    print('attn_weights', attn_weights.size())
    
    
    mask = torch.ones(seq_len, seq_len, dtype=torch.bool).tril(diagonal=0).view(1, 1, seq_len, seq_len)
    # mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    print('mask', mask)
    
    attn_weights = torch.where(mask == True, attn_weights, float('-inf'))
    
    # attn_weights = torch.masked_fill(attn_weights, mask, float('-inf'))
    
    attn_weights = torch.nn.functional.softmax(attn_weights)
    
    print('after softmax', attn_weights)
    
    attn_result = attn_weights @ v
    
    attn_result = attn_result.view(batch_size, seq_len, self.emb_dim)
    
    # print('attn_result', attn_result.size())
    
    
    

    return self.output_proj(attn_result)
  
  
  
  

x = torch.randn(batch_size, seq_len, emb_dim).to(device)
SelfAttn(emb_dim=emb_dim).to(device)(x).size()

attn_weights torch.Size([2, 5, 4, 4])
mask tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])


RuntimeError: The size of tensor a (5) must match the size of tensor b (4) at non-singleton dimension 3

torch.Size([2, 5])